In [ ]:
import sys
import os
# تنظیم متغیر محیطی برای غیرفعال کردن هشدار oneDNN در TensorFlow
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

from src.root import get_root
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from data_selector import Data_selector
from feature_adder import Feature_adder
from feature_selector import Feature_selector
from logs.logger import CustomLogger
from main import *
from models import Random_Forest, Linear, Polynomial, XGBoost, LinearL1

# تنظیمات نمایش داده‌های pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# راهنما: انتخاب ویژگی‌ها و دریافت X و y از دیتافریم
def select_features_and_get_X_and_y(df, features_to_be_select, is_mimo=False, number_mimo=None):
    """
    انتخاب ویژگی‌ها و استخراج ماتریس ورودی X و بردار هدف y از دیتافریم

    پارامترها:
    -----------
    df : pandas.DataFrame
        دیتافریمی که داده‌ها در آن هستند.
    features_to_be_select : list of str
        لیست نام ویژگی‌هایی که باید انتخاب شوند.
    is_mimo : bool, اختیاری
        مشخص می‌کند آیا مدل چند ورودی چند خروجی (MIMO) است یا خیر.
    number_mimo : int, اختیاری
        تعداد خروجی‌های MIMO در صورت فعال بودن.

    خروجی:
    -------
    X : numpy.ndarray
        ماتریس ویژگی‌ها
    y : numpy.ndarray
        بردار یا ماتریس هدف
    """
    feature_selector = Feature_selector(df, target="generation")
    # اضافه کردن ویژگی با تاخیر 24 ساعته به لیست ویژگی‌ها
    features_to_be_select.append(f"generation_with_{24}_delay")
    feature_selector.select(features_to_select=features_to_be_select)
    X, y = feature_selector.get_X_and_y(is_mimo=is_mimo, number_mimo=number_mimo)
    return X, y

# ساخت مدل شبکه عصبی 4 لایه با تابع فعال‌سازی ReLU
'''
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='linear'))  # لایه خروجی برای رگرسیون

# کامپایل مدل با Adam و خطای میانگین مربعات
model.compile(optimizer='adam', loss='mean_squared_error')
'''
#model.save(get_root() + '/models/model.model')
# آموزش مدل
# model.fit(X_train, y_train, epochs=100 , batch_size=32)
# ذخیره مدل در مسیر دلخواه
#model.save(get_root() + '/models/model.model')
#from tensorflow.keras.models import load_model
#model = load_model("my_model")


flag = 0

if flag == 0:#__name__ == "__main__":
    # TODO: for mimo > 1 doesn't work
    write_predictions = False

    number_mimo = 1
    is_mimo = number_mimo > 1
    y_is_flat = not is_mimo

    l_min = 4
    max_diff = 3
    c_thresh = 0.9

    df = add_features_and_filter(l_min, max_diff, c_thresh, read_from_integrated=False)
    logger.info(f"Csv file has bean labeled successfully")

    
    
    # تعریف ویژگی‌های انتخابی اولیه
    features_to_be_select = [
        "name", "code", "temperature", "humidity", "dew", "surface_pressure", "value", "forecast",
        "status", "season", "datetime"
    ]

    # افزودن ویژگی‌های تاخیر برای تعدادی از ویژگی‌ها
    space_features = ["temperature", "humidity", "dew", "surface_pressure", "value", "forecast", "status"]
    fa = Feature_adder(df, add_label_column=False)
    for feature in space_features:
        for i in range(24):
            fa.create_feature_with_delay(feature, i + 1)
            features_to_be_select.append(f"{feature}_with_{i + 1}_delay")


    ds = Data_selector(df)
    df_modified = ds.select_peaks(goodness=3)
    logger.info(f"Rows have been selected successfully")

    X, y = select_features_and_get_X_and_y(df_modified, features_to_be_select, is_mimo=is_mimo, number_mimo=number_mimo)
    logger.info(f"Some features have been dropped successfully")

    model = Random_Forest(n_estimators=1000, max_depth=1000)
    # model = Linear()
    # model = Polynomial(degree=2)
    # model = XGBoost(n_estimators=1000, max_depth=5)
    # model = Neural_network(input_dim=X.shape[1], epochs=100, verbose=1)
    model.scale_and_split_data(X, y, y_is_flat=y_is_flat)
    model.fit()
    logger.info(f"Model has been trained successfully")

    test_model(model)

    if write_predictions:
        write_result(df, model, X)

2025-09-22 18:21:11 - model_main - INFO - Csv file has bean labeled successfully
2025-09-22 18:21:24 - model_main - INFO - Rows have been selected successfully
2025-09-22 18:21:24 - model_main - INFO - Some features have been dropped successfully
2025-09-22 18:48:04 - model_main - INFO - Model has been trained successfully
2025-09-22 18:48:25 - model_main - INFO - Train Error: 0.79%, Test Error: 2.12%


idea : ditect and delete bad point in data

idea : add status with 1,2,3,..n delay

In [4]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_selector import Data_selector
from feature_adder import Feature_adder
from feature_selector import Feature_selector
from logs.logger import CustomLogger
from models import Random_Forest, Linear, Polynomial, XGBoost, LinearL1
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)
logger = CustomLogger(name="souri.test", log_file_name='souri_test.log').get_logger()
from main import *

In [5]:
def select_features_and_get_X_and_y(df, features_to_be_select,is_mimo=False, number_mimo=None):
    feature_selector = Feature_selector(df, target="generation")
    #for hour in range(1, 4):
    #    features_to_be_select.append(f"generation_with_{hour}_delay")
    features_to_be_select.append(f"generation_with_{24}_delay")
    feature_selector.select(features_to_select=features_to_be_select)
    X, y = feature_selector.get_X_and_y(is_mimo=is_mimo, number_mimo=number_mimo)
    return X, y

In [6]:
# TODO: for mimo > 1 doesn't work
write_predictions = False

number_mimo = 1
is_mimo = number_mimo > 1
y_is_flat = not is_mimo

l_min = 4
max_diff = 3
c_thresh = 0.9

df = add_features_and_filter(l_min, max_diff, c_thresh, read_from_integrated=False)
logger.info(f"Csv file has bean labeled successfully")


2025-09-22 18:48:26 - model_main - INFO - Csv file has bean labeled successfully


In [7]:
features_to_be_select = ["name", "code", "temperature", "humidity", "dew", "surface_pressure", "value", "forecast",
                             "status", "season"] + ["datetime"]

space_features = [ "temperature", "humidity", "dew", "surface_pressure", "value", "forecast","status"]
fa = Feature_adder(df,add_label_column=False)

for feature in space_features:
    for i in range(3):
        fa.create_feature_with_delay(feature,i+1)
        features_to_be_select.append(f"{feature}_with_{i+1}_delay")

In [8]:
ds = Data_selector(df)
df_modified = ds.select_peaks(goodness=3)
logger.info(f"Rows have been selected successfully")
X, y = select_features_and_get_X_and_y(df_modified,features_to_be_select, is_mimo=is_mimo, number_mimo=number_mimo)
logger.info(f"Some features have been dropped successfully")

2025-09-22 18:48:39 - model_main - INFO - Rows have been selected successfully
2025-09-22 18:48:39 - model_main - INFO - Some features have been dropped successfully


In [9]:
# model = Random_Forest(n_estimators=100, max_depth=1000)
# model = Linear()
# model = Polynomial(degree=2)
# model = XGBoost(n_estimators=1000, max_depth=5)
# model = Neural_network(input_dim=X.shape[1], epochs=100, verbose=1)

model = XGBoost(n_estimators=1200, max_depth=10)
model.scale_and_split_data(X, y, y_is_flat=y_is_flat)
model.fit()
logger.info(f"")
test_model(model)

if write_predictions:
    write_result(df, model, X)

2025-09-22 18:48:51 - model_main - INFO - 
2025-09-22 18:48:51 - model_main - INFO - Train Error: 0.16%, Test Error: 1.57%


XGBoost max_depth = 10 n_estimators = 1200 => Train Error: 0.16%, Test Error: 1.57%

XGBoost max_depth n_estimators = 1000

max_depth : 5  Train Error: 1.46%, Test Error: 1.93%

max_depth : 7  Train Error: 0.81%, Test Error: 1.68%

max_depth : 8  Train Error: 0.56%, Test Error: 1.62%

max_depth : 9  Train Error: 0.37%, Test Error: 1.61%

max_depth : 10 Train Error: 0.22%, Test Error: 1.58%

max_depth : 11 Train Error: 0.12%, Test Error: 1.61%

max_depth : 12 Train Error: 0.07%, Test Error: 1.64%

max_depth : 14 Train Error: 0.06%, Test Error: 1.71%

XGBoost max_depth = 10 n_estimators

n_estimators : 1000 Train Error: 0.22%, Test Error: 1.58%

n_estimators : 1200 Train Error: 0.16%, Test Error: 1.57%

n_estimators : 1400 Train Error: 0.13%, Test Error: 1.57%

n_estimators : 1600 Train Error: 0.10%, Test Error: 1.57%

n_estimators : 1800 Train Error: 0.09%, Test Error: 1.57%

In [10]:
import tensorflow
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential

model = Sequential()
model.add(Input(shape=(self.input_dim,)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='linear'))

model.compile(loss='mean_squared_error', optimizer='adam')

model.fit(self.X_train, self.y_train, epochs=self.epochs, verbose=self.verbose)

self.model_info = {
    "epochs": self.epochs,
}
self.model = model
logger.debug("Model trained successfully.")

ImportError: Traceback (most recent call last):
  File "C:\Users\alireza\AppData\Roaming\Python\Python310\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load sample data (Iris dataset)
#iris = load_iris()
#X = iris.data
#y = iris.target

param_grid = {
    'max_depth': [10,20],
    'learning_rate': [0.01],
    'n_estimators': [2000,4000],
}
model = xgb.XGBRegressor(objective='reg:squarederror')

def get_best_model(model,param_grid,X,y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, 
                            scoring='neg_mean_squared_error', cv=3, verbose=1)
    grid_search.fit(X_train, y_train)
    print("Best parameters:", grid_search.best_params_)
    print("Best accuracy:", grid_search.best_score_)
    best_model = grid_search.best_estimator_
    test_accuracy = best_model.score(X_test, y_test)
    print("Test set accuracy:", test_accuracy)
    
    return grid_search.best_estimator_

model = get_best_model(model,param_grid,X,y)

In [ ]:
grid_search.best_params_

In [ ]:
fa.create_feature_with_delay()

In [ ]:
y_pred_test = model.model.predict(model.X_test)
y_pred_train = model.model.predict(model.X_train)

y_pred_test_actual  = model.inverse_scale_array(model.scaler_y, y_pred_test)
y_pred_train_actual = model.inverse_scale_array(model.scaler_y, y_pred_train)
y_test_actual       = model.inverse_scale_array(model.scaler_y, model.y_test)
y_train_actual      = model.inverse_scale_array(model.scaler_y, model.y_train)

In [ ]:
y_diff = y_pred_train_actual - y_train_actual

In [ ]:
y_diff1 = np.linalg.norm(y_diff, axis=1)/np.linalg.norm(y_train_actual, axis=1)
ar = np.argsort(y_diff1)
y_hist = y_diff1[ar]

In [ ]:
ar[-43:]

In [ ]:
np.count_nonzero(y_hist<0.04)

In [ ]:
len(y_hist)

In [ ]:
plt.plot(np.diff(y_hist)[45300:45408])

In [ ]:
_ = plt.hist(y_hist[:-200],bins=1000)
plt.show()
_ = plt.hist(y_hist[-200:],bins=10)